# Feature Extraction

Proses ini bertujuan untuk mengonversi teks yang sudah bersih menjadi representasi vektor angka agar dapat diproses oleh algoritma Machine Learning. Teknik yang digunakan meliputi **TF-IDF**, **Word2Vec**, **FastText**, dan **GloVe**.

## 1. Persiapan Data
Memuat dataset yang telah dibersihkan pada tahap preprocessing.

In [2]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

# Unduh punkt untuk tokenisasi
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load data
df = pd.read_csv('../Dataset/cleaned_data.csv')

# Pastikan tidak ada missing values pada teks
docs = df['clean_text'].fillna('').astype(str).tolist()

# Mapping label ke numerik
y = df['label'].map({'real': 0, 'fake': 1}).values

print(f"Total Data: {len(docs)}")


Total Data: 20000


## 2. Ekstraksi Fitur: TF-IDF
**Term Frequency-Inverse Document Frequency (TF-IDF)** mengukur seberapa penting sebuah kata dalam suatu dokumen terhadap seluruh corpus.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Menggunakan max_features untuk membatasi dimensi agar tidak terlalu besar
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(docs)

vocab_tfidf = tfidf.get_feature_names_out()
print("Vocabulary Size TF-IDF:", len(vocab_tfidf))
print("TF-IDF Matrix Shape:", X_tfidf.shape)


Vocabulary Size TF-IDF: 869
TF-IDF Matrix Shape: (20000, 869)


## 3. Ekstraksi Fitur: Word Embedding (Word2Vec)
**Word2Vec** mempelajari representasi vektor dari kata berdasarkan konteks kemunculannya (menggunakan metode Skip-gram/CBOW).

In [4]:
from gensim.models import Word2Vec

# Tokenisasi dokumen untuk embedding
tokenized_docs = [word_tokenize(doc) for doc in docs]

# Melatih model Word2Vec dari awal
w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1, # Skip-gram
    epochs=10,
    workers=4
)

# Fungsi untuk membuat rata-rata vektor kata dalam satu dokumen
def doc_vector(tokens, model_wv, vector_size):
    vecs = [model_wv[w] for w in tokens if w in model_wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(vector_size)

# Representasi dokumen dengan Word2Vec
X_w2v = np.array([doc_vector(doc, w2v_model.wv, 100) for doc in tokenized_docs])
print("Word2Vec Matrix Shape:", X_w2v.shape)


Word2Vec Matrix Shape: (20000, 100)


## 4. Ekstraksi Fitur: Word Embedding (FastText)
**FastText** mirip dengan Word2Vec namun merepresentasikan kata sebagai kumpulan n-gram karakter, sehingga dapat menangani kata-kata yang tidak ada dalam vocabulary (Out-Of-Vocabulary).

In [5]:
from gensim.models import FastText

# Melatih model FastText
ft_model = FastText(
    sentences=tokenized_docs,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    epochs=10,
    workers=4
)

# Representasi dokumen dengan FastText
X_ft = np.array([doc_vector(doc, ft_model.wv, 100) for doc in tokenized_docs])
print("FastText Matrix Shape:", X_ft.shape)


FastText Matrix Shape: (20000, 100)


## 5. Ekstraksi Fitur: Word Embedding (GloVe)
**GloVe (Global Vectors for Word Representation)** memanfaatkan probabilitas kemunculan bersama kata dalam corpus secara global. Di sini kita menggunakan pre-trained GloVe model dari library Gensim.

In [6]:
import gensim.downloader as api

# Mengunduh dan memuat pre-trained GloVe (glove-wiki-gigaword-100)
# Catatan: Proses ini mungkin memakan waktu beberapa menit dan butuh memori karena mengunduh model ~128MB.
print("Loading pre-trained GloVe model...")
glove_model = api.load('glove-wiki-gigaword-100')

# Representasi dokumen dengan GloVe
X_glove = np.array([doc_vector(doc, glove_model, 100) for doc in tokenized_docs])
print("GloVe Matrix Shape:", X_glove.shape)


Loading pre-trained GloVe model...
[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe Matrix Shape: (20000, 100)


## 6. Penyimpanan Hasil Ekstraksi Fitur
Menyimpan matriks fitur dan label ke dalam file untuk digunakan pada tahap Modeling.

In [7]:
from scipy import sparse
import os

# Menyimpan di folder Notebook atau Dataset (kita buat folder features untuk merapihkan)
out_dir = 'extracted_features'
os.makedirs(out_dir, exist_ok=True)

# Simpan TF-IDF sebagai sparse matrix untuk menghemat ruang
sparse.save_npz(f'{out_dir}/X_tfidf.npz', X_tfidf)

# Simpan Word Embeddings dan Labels sebagai numpy array (.npy)
np.save(f'{out_dir}/X_w2v.npy', X_w2v)
np.save(f'{out_dir}/X_ft.npy', X_ft)
np.save(f'{out_dir}/X_glove.npy', X_glove)
np.save(f'{out_dir}/y_labels.npy', y)

print(f"Semua fitur berhasil disimpan di dalam folder '{out_dir}/'")


Semua fitur berhasil disimpan di dalam folder 'extracted_features/'
